In [ ]:
import os, re, ast, math, json, asyncio, warnings
from pathlib import Path
from typing import List, Dict, Optional
from openai import OpenAI
from ragas import RunConfig, evaluate, EvaluationDataset, SingleTurnSample
import httpx
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
from rouge_score import rouge_scorer
from sentence_transformers import util
from neo4j import GraphDatabase
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness, FactualCorrectness,
    ResponseGroundedness, ContextRelevance, AnswerAccuracy
)
from dataclasses import dataclass
from typing import List
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings("ignore")
load_dotenv()
os.environ.get("OPENAI_API_KEY")

LLM_MODEL = "gpt-4o-mini"

_chat_llm = ChatOpenAI(
    model=LLM_MODEL,
    max_tokens=16384
)
eval_llm_high = LangchainLLMWrapper(_chat_llm)

In [ ]:
import nest_asyncio
nest_asyncio.apply()

USER_INPUT = "Co pokazują statystyki rynku pracy w 2026?"

RESPONSE = (
    ""
)

REFERENCE = (
"""



"""
)

RETRIEVED_CONTEXTS = [
    """"""
]

REFERENCE_CONTEXTS = [
    """"""
]

METRICS_TO_RUN = [
    "faithfulness",
    "factual_correctness",
    "response_groundedness",
    "context_relevance",
    "answer_accuracy",
]

def build_metrics(llm_wrapper):
    all_metrics = {
        "faithfulness": Faithfulness(llm=llm_wrapper),
        "factual_correctness": FactualCorrectness(llm=llm_wrapper),
        "response_groundedness": ResponseGroundedness(llm=llm_wrapper),
        "context_relevance": ContextRelevance(llm=llm_wrapper),
        "answer_accuracy": AnswerAccuracy(llm=llm_wrapper),
    }
    return [all_metrics[k] for k in METRICS_TO_RUN if k in all_metrics]


metrics = build_metrics(eval_llm_high)

sample = SingleTurnSample(
    user_input=USER_INPUT,
    response=RESPONSE,
    reference=REFERENCE,
    retrieved_contexts=RETRIEVED_CONTEXTS,
    reference_contexts=REFERENCE_CONTEXTS,
)

run_cfg = RunConfig(max_retries=10, max_wait=160, timeout=220)

result = evaluate(
    dataset=EvaluationDataset(samples=[sample]),
    metrics=metrics,
    run_config=run_cfg,
    raise_exceptions=False,
)

df = result.to_pandas()
row = df.iloc[0]

print("\n" + "=" * 55)
print(f"  Question:  {USER_INPUT}")
print(f"  Answer:    {RESPONSE[:80]}{'...' if len(RESPONSE) > 80 else ''}")
print("=" * 55)
for col in df.columns:
    val = row[col]
    if isinstance(val, float) and not math.isnan(val):
        print(f"  {col:<40} {val:.4f}")
print("=" * 55)

In [ ]:
sample = SingleTurnSample(
    user_input=USER_INPUT,
    response=RESPONSE,
    reference=REFERENCE,
    retrieved_contexts=[REFERENCE],
    reference_contexts=[REFERENCE],
)
dataset = EvaluationDataset(samples=[sample])
run_cfg = RunConfig(max_retries=5, max_wait=30, timeout=60)

for mode in ("precision", "recall", "f1"):
    metric = FactualCorrectness(llm=eval_llm_high, mode=mode)
    result = evaluate(dataset=dataset, metrics=[metric],
                      run_config=run_cfg, raise_exceptions=False)
    df = result.to_pandas()
    val = df.iloc[0][df.columns[-1]]
    print(f"  factual_correctness(mode={mode}) = {val:.4f}")